# Cài đặt thư viện

In [ ]:
!pip install -q datasets==3.2.0
!pip install --upgrade huggingface-hub evaluate

# Load dataset

In [ ]:
from datasets import load_dataset
ds = load_dataset("thainq107/abte-restaurants")

ds

# Load Tokenizer

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

# Tiền xử lý dữ liệu

In [ ]:
def tokenize_and_align_labels(examples):
    sentences, sentence_tags = [], []
    labels = []
    for tokens, pols in zip(examples['Tokens'], examples['Polarities']):
        bert_tokens = []
        bert_att = []
        pols_label = 0
        for i in range(len(tokens)):
            t = tokenizer.tokenize(tokens[i])
            bert_tokens += t
            if int(pols[i]) != -1:
                bert_att += t
                pols_label = int(pols[i])

        sentences.append(" ".join(bert_tokens))
        sentence_tags.append(" ".join(bert_att))
        labels.append(pols_label)
    
    tokenized_inputs = tokenizer(sentences, sentence_tags, padding=True, truncation=True, return_tensors="pt")
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

In [ ]:
preprocessed_ds = ds.map(tokenize_and_align_labels, batched=True)
preprocessed_ds

# Load Model

In [ ]:
from transformers import AutoModelForSequenceClassification

id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}
label2id = {"Negative": 0, "Neutral": 1, "Positive": 2}

model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert/distilbert-base-uncased",
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
model

# Huấn luyện mô hình

In [ ]:
import numpy as np
import evaluate

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return accuracy.compute(predictions=predictions, references=labels)

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./absa-restaurant-albert-base-v2",
    learning_rate=2e-5,
    per_device_train_batch_size=128,
    per_device_eval_batch_size=128,
    num_train_epochs=50,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=preprocessed_ds["train"],
    eval_dataset=preprocessed_ds["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()

# Kiểm thử mô hình

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import torch
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

atsc_model = AutoModelForSequenceClassification.from_pretrained("./absa-restaurant-albert-base-v2/checkpoint-1450")

id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}

def absa_inference(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)

    with torch.no_grad():
        outputs = atsc_model(**inputs)
        logits = outputs.logits
        predicted_class_id = np.argmax(logits.detach().cpu().numpy())
        sentiment = id2label[predicted_class_id]
        score = torch.softmax(logits, dim=1)[0][predicted_class_id].item()
    return {"text": text, "sentiment": sentiment, "score": score}

text = "But the staff was so horriable to us."
result = absa_inference(text)
print(result)


In [ ]:
import os
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from huggingface_hub import login

# 1. Dang nhap Hugging Face
login()

# 2. Cau hinh
base_dir = "./absa-restaurant-albert-base-v2"
tokenizer_name = "distilbert/distilbert-base-uncased"
repo_id = "GinNoV111/absa-restaurant-model"   

# 3. Checkpoint moi nhat
checkpoint_path = "./absa-restaurant-albert-base-v2/checkpoint-1450" 

# 4. Load model va tokenizer
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_path)
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

id2label = {0: "Negative", 1: "Neutral", 2: "Positive"}
label2id = {v: k for k, v in id2label.items()}

model.config.id2label = id2label
model.config.label2id = label2id

# 5. Push len Hugging Face Hub
model.push_to_hub(repo_id)
tokenizer.push_to_hub(repo_id)

print(f"Da push model len: https://huggingface.co/{repo_id}")
